# Five-microgrid ablation study: trust-region samples vs. global random samples

This notebook runs one complete experiment from top to bottom. It does not load an old trust-region run. The four methods compared at the end are:

1. the original global L2O models solved with the NN solver;
2. trust-region residual correction initialized from the original NN solution;
3. global L2O models retrained on original + trust-region samples;
4. global L2O models retrained on original + equally many global random samples.

The full run is expensive because it includes five lower-level models, a trust-region run, two rounds of training, and three 10,000-step NN solves.

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pandapower.networks as pn

import follower_solver
from L2Otraining import TrainingConfig, train
from stackelberg.followers.bg import create_cigre_residential_network
from stackelberg.followers.ip import create_industrial_park_network
from stackelberg.instances import resolve_follower_configuration
from stackelberg.network import create_case33bw_20kv
from stackelberg.nn_solver import (
    NNSolverExperimentConfig,
    run_nn_solver_experiment,
)
from stackelberg.randomness import make_rng, set_global_seed
from stackelberg.trust_region import (
    TrustRegionExperimentConfig,
    run_trust_region_experiment,
)

PROJECT_ROOT = Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "NNmodel"
RESULTS_DIR = PROJECT_ROOT / "results"

if not (PROJECT_ROOT / "L2Otraining.py").is_file():
    raise RuntimeError("Run this notebook with the project root as the working directory.")

DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

T = 24
SEED = 42
NN_STEPS = 10_000
NN_LEARNING_RATE = 1e-2
NN_SAVE_EVERY = 200
TRAINING_EPOCHS = 500
LOWER_TIME_LIMIT = 30.0
LOWER_MIP_GAP = 1e-3
NEW_DATA_CHECKPOINT_INTERVAL = 100
NEW_DATA_SEEDS = {
    "SB1": 4201,
    "SB2": 4202,
    "BG1": 4203,
    "BG2": 4204,
    "IP1": 4205,
}

SCENARIOS = ["SB1", "SB2", "BG1", "BG2", "IP1"]
MG_BUSES = [17, 32, 15, 21, 1]
UPPER_NET = create_case33bw_20kv()
LOWER_NETS = [
    pn.create_cigre_network_lv(),
    pn.create_cigre_network_lv(),
    create_cigre_residential_network(),
    create_cigre_residential_network(),
    create_industrial_park_network(),
]
X0 = np.full((len(SCENARIOS), T), 0.75, dtype=float)

follower_ids, scenario_types, follower_kwargs = resolve_follower_configuration(
    SCENARIOS
)
if follower_ids != SCENARIOS:
    raise RuntimeError(f"Resolved follower order {follower_ids} does not match {SCENARIOS}.")

INSTANCE_NETWORKS = dict(zip(SCENARIOS, LOWER_NETS))
INSTANCE_TYPES = dict(zip(SCENARIOS, scenario_types))
INSTANCE_SOLVER_KWARGS = dict(zip(SCENARIOS, follower_kwargs))

instance_table = pd.DataFrame(
    {
        "follower_id": SCENARIOS,
        "scenario_type": scenario_types,
        "mg_bus": MG_BUSES,
        "solver_kwargs": follower_kwargs,
    }
).set_index("follower_id")
display(instance_table)

,scenario_type,mg_bus,solver_kwargs
follower_id,,,
SB1,SB,17,{}
SB2,SB,32,{'load_scale': 1.1}
BG1,BG,15,{'date': '2023-01-01'}
BG2,BG,21,{'date': '2023-07-01'}
IP1,IP,1,{'date': '2023-01-01'}


In [2]:
def load_aligned_arrays(x_path, y_path, *, expected_rows=None, expected_width=T):
    """Load and validate one aligned price/response dataset."""
    x_path = Path(x_path)
    y_path = Path(y_path)
    if not x_path.is_file() or not y_path.is_file():
        raise FileNotFoundError(f"Expected both {x_path} and {y_path}.")

    x_data = np.load(x_path)
    y_data = np.load(y_path)
    if x_data.ndim != 2 or y_data.ndim != 2:
        raise ValueError(f"Expected 2-D arrays, got X{x_data.shape} and Y{y_data.shape}.")
    if x_data.shape != y_data.shape:
        raise ValueError(f"X/Y shape mismatch: X{x_data.shape} and Y{y_data.shape}.")
    if x_data.shape[1] != expected_width:
        raise ValueError(f"Expected width {expected_width}, got {x_data.shape[1]}.")
    if len(x_data) == 0:
        raise ValueError(f"Dataset is empty: {x_path.name}.")
    if expected_rows is not None and len(x_data) != expected_rows:
        raise ValueError(
            f"Expected {expected_rows} rows in {x_path.name}, got {len(x_data)}."
        )
    if not np.isfinite(x_data).all() or not np.isfinite(y_data).all():
        raise ValueError(f"Non-finite values found in {x_path.name} or {y_path.name}.")
    if np.any(x_data < -1e-12) or np.any(x_data > 1.5 + 1e-12):
        raise ValueError(f"Prices in {x_path.name} must lie in [0, 1.5].")
    return x_data, y_data


def atomic_save_npy(path, array):
    """Replace a NumPy file only after the complete array is serialized."""
    path = Path(path)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    with temporary_path.open("wb") as file:
        np.save(file, np.asarray(array))
    temporary_path.replace(path)


for scenario in SCENARIOS:
    load_aligned_arrays(
        DATA_DIR / f"X_{scenario}.npy",
        DATA_DIR / f"Y_{scenario}.npy",
    )
    for suffix in (".pth", ".json"):
        model_path = MODEL_DIR / f"L2O_{scenario}{suffix}"
        if not model_path.is_file():
            raise FileNotFoundError(model_path)

print("Validated original datasets and L2O checkpoints for all five instances.")

Validated original datasets and L2O checkpoints for all five instances.


## 1. Original global models and NN solver

All three NN-solver runs use the same initial price matrix and the same 10,000-step optimization budget.

In [3]:
original_nn_config = NNSolverExperimentConfig(
    upper_net=UPPER_NET,
    lower_nets=LOWER_NETS,
    mg_buses=MG_BUSES,
    scenarios=SCENARIOS,
    T=T,
    seed=SEED,
    device="cpu",
    x0=X0,
    steps=NN_STEPS,
    lr=NN_LEARNING_RATE,
    save_every=NN_SAVE_EVERY,
    history_path=str(RESULTS_DIR / "NN_solver_original_ablation_history.csv"),
    evaluate_true=True,
    save_results=True,
)
original_nn_result = run_nn_solver_experiment(original_nn_config)
original_nn_prices = np.asarray(
    original_nn_result["optimal pi"], dtype=float
).reshape(len(SCENARIOS), T)

print("Original NN history:", original_nn_result["history_path"])
print("Original NN true objective:", original_nn_result["true_obj"])
print("Original NN runtime (s):", original_nn_result["runtime_seconds"])
display(pd.DataFrame(original_nn_prices, index=SCENARIOS))

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

[0] loss=5821.5625, obj=-123.5207, eq=5.9380, soc=0.0647, bound=0.0646, true_loss=-0.1048
Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copy

,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
SB1,0.970181,0.972706,0.975164,1.000694,0.959616,0.969702,1.160907,1.497205,1.497323,0.998421,...,1.047077,1.499775,1.496677,1.499511,1.337253,1.495676,1.489284,1.496565,1.093230,0.966810
SB2,0.984327,1.489236,0.971531,0.980285,0.946286,0.950465,0.998462,0.976595,1.498717,1.498230,...,1.016355,1.498472,1.497773,1.498430,1.498577,1.499142,1.497796,1.451597,1.440844,1.184781
BG1,1.499816,1.499887,1.499895,1.499872,1.499943,1.499931,1.499945,1.499928,1.499881,1.499913,...,1.499950,1.499931,1.499941,1.499935,1.499918,1.499891,1.499909,1.499468,1.499928,1.499742
BG2,1.499647,1.499705,1.499627,1.499876,1.499915,1.499792,1.499918,1.499930,1.499858,1.499912,...,1.499942,1.499920,1.499952,1.499877,1.499927,1.499919,1.499869,1.499671,1.499878,1.499435
IP1,1.496455,1.492518,1.494619,1.495886,1.491641,1.457786,1.486227,1.497947,1.498367,1.497680,...,1.498705,1.498223,1.498699,1.496387,1.490192,1.497343,1.496753,1.495072,1.498292,1.497317


## 2. Trust region and fresh trust-region training samples

The trust region starts at the original NN solution. `save_true_responses=True` retains the high-fidelity residual-fit designs, including each local center, and the controller removes duplicate rows across iterations before export. The resulting files replace the known-invalid `_tr` datasets.

In [4]:
tr_config = TrustRegionExperimentConfig(
    upper_net=UPPER_NET,
    lower_nets=LOWER_NETS,
    mg_buses=MG_BUSES,
    scenarios=SCENARIOS,
    T=T,
    seed=SEED,
    device="cpu",
    x0=original_nn_prices,
    lower_time_limit=LOWER_TIME_LIMIT,
    lower_mip_gap=LOWER_MIP_GAP,
    tr_params={
        "R_0": 0.05,
        "R_max": 1.0,
        "gamma_dec": 0.7,
        "gamma_inc": 1.5,
        "eta_1": 0.1,
        "eta_2": 0.75,
    },
    algo_params={
        "n_samples": 60,
        "K_max": 60,
        "sampling_method": "lhs",
        "residual_update_mode": "accumulate",
        "rbf_geometry_condition_max": 1e12,
        "solve_params": {
            "steps_init": 0,
            "steps_warm": 2000,
            "lr": 1e-2,
            "penalty_weights": {
                "eq": 1000.0,
                "ineq": 100.0,
                "soc": 100.0,
                "bound": 10.0,
            },
            "tr_weight": 1e6,
            "verbose": True,
        },
    },
    residual_model_params={
        "model_type": "rbf",
        "rbf_kernel": "cubic",
        "rbf_smoothing": 1e-3,
        "rbf_degree": 1,
        "rbf_scale": "trust_radius",
        "rbf_epsilon": None,
    },
    save_results=True,
    output_dir=str(RESULTS_DIR),
    save_true_responses=True,
    true_response_dir=str(DATA_DIR),
    true_response_tag="tr",
)
tr_result = run_trust_region_experiment(tr_config)
tr_prices = np.asarray(tr_result["x_opt"], dtype=float).reshape(
    len(SCENARIOS), T
)
tr_history = pd.DataFrame(tr_result["history"])
if tr_history.empty:
    raise RuntimeError("Trust-region history is empty.")
tr_last = tr_history.iloc[-1]
tr_objective = float(
    tr_last["F_candidate"]
    if str(tr_last["decision"]).startswith("accept")
    else tr_last["F_current"]
)
tr_composite_queries = int(
    tr_result["n_iter"] * (tr_config.algo_params["n_samples"] + 1)
)

print("Trust region converged:", tr_result["converged"])
print("Trust region iterations:", tr_result["n_iter"])
print("Trust region true objective:", tr_objective)
print("Trust region composite high-fidelity queries:", tr_composite_queries)
print("Saved paths:", tr_result.get("saved_paths"))

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Using global surrogate to solve initial point...
Initial surrogate solve used steps_init=0. Time taken to solve initial point 0.0488 seconds.
Iteration 0 | R=0.0500 
Building 60-point local designs for each of 5 followers (center + coordinate star + LHS) within R_tr=0.0500...
Fitting follower-local residual models with sample counts [60, 60, 60, 60, 60]...

In [5]:
current_tr_responses = tr_result.get("true_responses", {})
if set(current_tr_responses) != set(SCENARIOS):
    raise RuntimeError(
        f"Expected current-run TR samples for {SCENARIOS}, got "
        f"{sorted(current_tr_responses)}."
    )

tr_sample_counts = {}
sample_rows = []
for scenario in SCENARIOS:
    original_x, original_y = load_aligned_arrays(
        DATA_DIR / f"X_{scenario}.npy",
        DATA_DIR / f"Y_{scenario}.npy",
    )
    tr_x, tr_y = load_aligned_arrays(
        DATA_DIR / f"X_{scenario}_tr.npy",
        DATA_DIR / f"Y_{scenario}_tr.npy",
    )
    in_memory_x = np.asarray(current_tr_responses[scenario]["X"], dtype=float)
    in_memory_y = np.asarray(current_tr_responses[scenario]["Y"], dtype=float)
    np.testing.assert_allclose(tr_x, in_memory_x)
    np.testing.assert_allclose(tr_y, in_memory_y)

    tr_sample_counts[scenario] = len(tr_x)
    sample_rows.append(
        {
            "scenario": scenario,
            "original_samples": len(original_x),
            "trust_region_samples": len(tr_x),
            "tr_combined_samples": len(original_x) + len(tr_x),
        }
    )

tr_sample_count_table = pd.DataFrame(sample_rows).set_index("scenario")
display(tr_sample_count_table)

,original_samples,trust_region_samples,tr_combined_samples
scenario,,,
SB1,10000,1392,11392
SB2,10000,1342,11342
BG1,10000,1097,11097
BG2,10000,1064,11064
IP1,10000,1309,11309


## 3. Retrain global models on original + trust-region samples

Each follower instance remains separate. Training writes `L2O_XX_tr.pth/.json` and leaves the original checkpoints unchanged.

In [6]:
def training_config(scenario):
    return TrainingConfig(
        scenario=scenario,
        hidden_dim=128,
        residual_blocks=3,
        batch_size=64,
        learning_rate=1e-3,
        epochs=TRAINING_EPOCHS,
        validation_fraction=0.2,
        seed=SEED,
    )


tr_training_results = {}
for scenario in SCENARIOS:
    print(f"\nTraining {scenario} on original + trust-region samples")
    tr_training_results[scenario] = train(
        scenario,
        config=training_config(scenario),
        data_tag="tr",
        model_tag="tr",
        include_original=True,
    )
    for suffix in (".pth", ".json"):
        expected_model = MODEL_DIR / f"L2O_{scenario}_tr{suffix}"
        if not expected_model.is_file():
            raise FileNotFoundError(expected_model)

display(pd.DataFrame(tr_training_results).T.rename_axis("scenario"))


Training SB1 on original + trust-region samples
SB1: concatenated original 10000 + tr 1392 samples -> 11392
SB1 epoch   10/500: train=0.043312, validation=0.045060
SB1 epoch   20/500: train=0.032151, validation=0.033518
SB1 epoch   30/500: train=0.028143, validation=0.029543
SB1 epoch   40/500: train=0.021942, validation=0.022977
SB1 epoch   50/500: train=0.016209, validation=0.016810
SB1 epoch   60/500: train=0.010084, validation=0.010510
SB1 epoch   70/500: train=0.012549, validation=0.013118
SB1 epoch   80/500: train=0.019697, validation=0.020156
SB1 epoch   90/500: train=0.014559, validation=0.014865
SB1 epoch  100/500: train=0.040199, validation=0.040524
SB1 epoch  110/500: train=0.020302, validation=0.020560
SB1 epoch  120/500: train=0.007437, validation=0.007669
SB1 epoch  130/500: train=0.004328, validation=0.004538
SB1 epoch  140/500: train=0.005286, validation=0.005548
SB1 epoch  150/500: train=0.009669, validation=0.009763
SB1 epoch  160/500: train=0.003923, validation=0.00

,train_mse,validation_mse
scenario,,
SB1,0.000776,0.000830
SB2,0.000830,0.000938
BG1,0.000545,0.000597
BG2,0.000495,0.000549
IP1,0.777192,1.257545


In [14]:
tr_nn_config = NNSolverExperimentConfig(
    upper_net=UPPER_NET,
    lower_nets=LOWER_NETS,
    mg_buses=MG_BUSES,
    scenarios=SCENARIOS,
    T=T,
    seed=SEED,
    device="cpu",
    x0=X0,
    steps=20000,
    lr=NN_LEARNING_RATE,
    save_every=NN_SAVE_EVERY,
    history_path=str(RESULTS_DIR / "NN_solver_tr_ablation_history.csv"),
    evaluate_true=True,
    save_results=True,
    model_tag="tr",
)
tr_nn_result = run_nn_solver_experiment(tr_nn_config)
tr_nn_prices = np.asarray(tr_nn_result["optimal pi"], dtype=float).reshape(
    len(SCENARIOS), T
)

print("TR-retrained NN history:", tr_nn_result["history_path"])
print("TR-retrained NN true objective:", tr_nn_result["true_obj"])
print("TR-retrained NN runtime (s):", tr_nn_result["runtime_seconds"])

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

[0] loss=5804.0781, obj=-122.5196, eq=5.9195, soc=0.0647, bound=0.0646, true_loss=2.2705
Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyr

## 4. Generate an equal-size global-random control dataset

For each follower instance, prices are drawn independently and uniformly from $[0,1.5]^{24}$. The requested sample count is the actual number of unique trust-region rows exported for that same instance. Existing `_new` arrays are never loaded or reused.

In [8]:
def generate_fresh_random_samples(
    scenario,
    n_samples,
    *,
    seed,
    checkpoint_interval=100,
    max_consecutive_failures=20,
):
    """Generate a fresh deterministic `_new` dataset for one instance."""
    if n_samples <= 0:
        raise ValueError("n_samples must be positive.")

    set_global_seed(seed)
    rng = make_rng(seed)
    solver_options = {
        **INSTANCE_SOLVER_KWARGS[scenario],
        "failure_mode": "none",
        "include_metadata": True,
        "time_limit": LOWER_TIME_LIMIT,
        "mip_gap": LOWER_MIP_GAP,
    }
    lower_solver = follower_solver.build_solver(
        INSTANCE_TYPES[scenario],
        T=T,
        net=INSTANCE_NETWORKS[scenario],
        **solver_options,
    )

    x_path = DATA_DIR / f"X_{scenario}_new.npy"
    y_path = DATA_DIR / f"Y_{scenario}_new.npy"
    x_data = np.empty((n_samples, T), dtype=float)
    y_data = np.empty((n_samples, T), dtype=float)
    completed = 0
    attempts = 0
    failures = 0
    consecutive_failures = 0
    checkpoint_start = time.time()

    print(f"{scenario}: generating {n_samples} fresh samples (seed={seed}).")
    while completed < n_samples:
        attempts += 1
        price = rng.uniform(0.0, 1.5, size=T)
        result = lower_solver.lower_solve(price)
        if result is None:
            failures += 1
            consecutive_failures += 1
            print(
                f"{scenario}: skipped failed attempt {attempts}; "
                f"collected {completed}/{n_samples}."
            )
            if consecutive_failures >= max_consecutive_failures:
                raise RuntimeError(
                    f"{scenario}: stopped after {consecutive_failures} "
                    "consecutive lower-level failures."
                )
            continue

        consecutive_failures = 0
        response = np.asarray(result["p_mg"], dtype=float).reshape(T)
        if not np.isfinite(response).all():
            raise ValueError(f"{scenario}: follower returned a non-finite response.")
        x_data[completed] = price
        y_data[completed] = response
        completed += 1

        if completed % checkpoint_interval == 0 or completed == n_samples:
            atomic_save_npy(x_path, x_data[:completed])
            atomic_save_npy(y_path, y_data[:completed])
            elapsed = time.time() - checkpoint_start
            print(
                f"{scenario}: saved {completed}/{n_samples} fresh samples "
                f"({failures} failed attempts, {elapsed:.1f} s)."
            )
            checkpoint_start = time.time()

    return x_data, y_data


In [9]:
new_data = {}
for scenario in SCENARIOS:
    new_data[scenario] = generate_fresh_random_samples(
        scenario,
        tr_sample_counts[scenario],
        seed=NEW_DATA_SEEDS[scenario],
        checkpoint_interval=NEW_DATA_CHECKPOINT_INTERVAL,
    )

sample_rows = []
for scenario in SCENARIOS:
    original_x, _ = load_aligned_arrays(
        DATA_DIR / f"X_{scenario}.npy",
        DATA_DIR / f"Y_{scenario}.npy",
    )
    tr_x, _ = load_aligned_arrays(
        DATA_DIR / f"X_{scenario}_tr.npy",
        DATA_DIR / f"Y_{scenario}_tr.npy",
    )
    new_x, _ = load_aligned_arrays(
        DATA_DIR / f"X_{scenario}_new.npy",
        DATA_DIR / f"Y_{scenario}_new.npy",
        expected_rows=len(tr_x),
    )
    if len(new_x) != len(tr_x):
        raise AssertionError(
            f"{scenario}: new sample count {len(new_x)} != TR count {len(tr_x)}."
        )
    sample_rows.append(
        {
            "scenario": scenario,
            "original_samples": len(original_x),
            "trust_region_samples": len(tr_x),
            "new_random_samples": len(new_x),
            "tr_combined_samples": len(original_x) + len(tr_x),
            "new_combined_samples": len(original_x) + len(new_x),
        }
    )

sample_count_table = pd.DataFrame(sample_rows).set_index("scenario")
if not (
    sample_count_table["trust_region_samples"]
    == sample_count_table["new_random_samples"]
).all():
    raise AssertionError("Random and trust-region sample counts must match.")
display(sample_count_table)

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

SB1: generating 1392 fresh samples (seed=4201).
SB1: saved 100/1392 fresh samples (0 failed attempts, 1.8 s).
SB1: saved 200/1392 fresh samples (0 failed attempts, 1.7 s).
SB1: saved 300/1392 fresh samples (0 failed attempts, 1.7 s).
SB1: saved 400/1392 fresh samples (0 failed attempts, 1.6 s).
SB1: saved 500/1392 fresh samples (0 failed attempts, 1.7 s).
SB1: saved 600/1392 fresh samples (0 failed attempts, 1.7 s).
SB1: saved 700/1392 fresh samples (0 failed attempts, 1.7 s).
SB1: saved 800/1392 fresh samples (0 failed attempts, 1.8 s).
SB1: saved 900/1392 fresh samples (0 failed attempts, 1.8 s).
SB1: saved 1000/1392 fresh samples (0 failed attempts, 1.7 s).
SB1: saved 1100/1392 fresh samples (0 failed attempts, 1.6 s).
SB1: saved 1200/1392 fresh samples (0 failed attempts, 1.6 s).
SB1: saved 1300/1392 fresh samples (0 failed attempts, 1.6 s).
SB1: saved 1392/1392 fresh samples 

,original_samples,trust_region_samples,new_random_samples,tr_combined_samples,new_combined_samples
scenario,,,,,
SB1,10000,1392,1392,11392,11392
SB2,10000,1342,1342,11342,11342
BG1,10000,1097,1097,11097,11097
BG2,10000,1064,1064,11064,11064
IP1,10000,1309,1309,11309,11309


## 5. Retrain global models on original + equal-size random samples

Training writes `L2O_XX_new.pth/.json`, then the NN solver uses the same initial point and optimization budget as the other NN runs.

In [10]:
new_training_results = {}
for scenario in SCENARIOS:
    print(f"\nTraining {scenario} on original + equal-size global random samples")
    new_training_results[scenario] = train(
        scenario,
        config=training_config(scenario),
        data_tag="new",
        model_tag="new",
        include_original=True,
    )
    for suffix in (".pth", ".json"):
        expected_model = MODEL_DIR / f"L2O_{scenario}_new{suffix}"
        if not expected_model.is_file():
            raise FileNotFoundError(expected_model)

display(pd.DataFrame(new_training_results).T.rename_axis("scenario"))


Training SB1 on original + equal-size global random samples
SB1: concatenated original 10000 + new 1392 samples -> 11392
SB1 epoch   10/500: train=0.054344, validation=0.056812
SB1 epoch   20/500: train=0.037359, validation=0.039618
SB1 epoch   30/500: train=0.031813, validation=0.033871
SB1 epoch   40/500: train=0.024504, validation=0.026595
SB1 epoch   50/500: train=0.015438, validation=0.016905
SB1 epoch   60/500: train=0.012559, validation=0.013723
SB1 epoch   70/500: train=0.010040, validation=0.011200
SB1 epoch   80/500: train=0.009581, validation=0.010699
SB1 epoch   90/500: train=0.005985, validation=0.006853
SB1 epoch  100/500: train=0.004866, validation=0.005639
SB1 epoch  110/500: train=0.003688, validation=0.004289
SB1 epoch  120/500: train=0.003270, validation=0.003825
SB1 epoch  130/500: train=0.002841, validation=0.003403
SB1 epoch  140/500: train=0.002269, validation=0.002771
SB1 epoch  150/500: train=0.002699, validation=0.003192
SB1 epoch  160/500: train=0.001717, va

,train_mse,validation_mse
scenario,,
SB1,0.000735,0.000748
SB2,0.000595,0.000802
BG1,0.000441,0.000535
BG2,0.000468,0.000568
IP1,0.234742,0.757823


In [15]:
new_nn_config = NNSolverExperimentConfig(
    upper_net=UPPER_NET,
    lower_nets=LOWER_NETS,
    mg_buses=MG_BUSES,
    scenarios=SCENARIOS,
    T=T,
    seed=SEED,
    device="cpu",
    x0=X0,
    steps=20000,
    lr=NN_LEARNING_RATE,
    save_every=NN_SAVE_EVERY,
    history_path=str(RESULTS_DIR / "NN_solver_new_ablation_history.csv"),
    evaluate_true=True,
    save_results=True,
    model_tag="new",
)
new_nn_result = run_nn_solver_experiment(new_nn_config)
new_nn_prices = np.asarray(new_nn_result["optimal pi"], dtype=float).reshape(
    len(SCENARIOS), T
)

print("Random-retrained NN history:", new_nn_result["history_path"])
print("Random-retrained NN true objective:", new_nn_result["true_obj"])
print("Random-retrained NN runtime (s):", new_nn_result["runtime_seconds"])

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

[0] loss=6079.8252, obj=-123.3201, eq=6.1960, soc=0.0647, bound=0.0646, true_loss=1.2892
Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyr

## 6. Four-way solution and high-fidelity objective comparison

Lower true objective values are better. The key ablation comparison is between the two equally sized augmented training sets.

In [13]:
tr_saved_paths = tr_result.get("saved_paths", {})
comparison = pd.DataFrame(
    [
        {
            "method": "Original NN",
            "true_objective": float(original_nn_result["true_obj"]),
            "runtime_seconds": float(original_nn_result["runtime_seconds"]),
            "iterations": NN_STEPS,
            "composite_queries": np.nan,
            "history_or_result_path": original_nn_result["history_path"],
        },
        {
            "method": "Trust region",
            "true_objective": tr_objective,
            "runtime_seconds": float(tr_result["elapsed_seconds"]),
            "iterations": int(tr_result["n_iter"]),
            "composite_queries": tr_composite_queries,
            "history_or_result_path": tr_saved_paths.get("json"),
        },
        {
            "method": "NN: original + TR samples",
            "true_objective": float(tr_nn_result["true_obj"]),
            "runtime_seconds": float(tr_nn_result["runtime_seconds"]),
            "iterations": NN_STEPS,
            "composite_queries": np.nan,
            "history_or_result_path": tr_nn_result["history_path"],
        },
        {
            "method": "NN: original + equal global random samples",
            "true_objective": float(new_nn_result["true_obj"]),
            "runtime_seconds": float(new_nn_result["runtime_seconds"]),
            "iterations": NN_STEPS,
            "composite_queries": np.nan,
            "history_or_result_path": new_nn_result["history_path"],
        },
    ]
).set_index("method")

if len(comparison) != 4:
    raise AssertionError(f"Expected four comparison rows, got {len(comparison)}.")
if not np.isfinite(comparison["true_objective"].to_numpy(dtype=float)).all():
    raise ValueError("All four high-fidelity objectives must be finite.")

hour_columns = [f"hour_{hour:02d}" for hour in range(T)]
solution_frames = {
    "Original NN": pd.DataFrame(
        original_nn_prices, index=SCENARIOS, columns=hour_columns
    ),
    "Trust region": pd.DataFrame(
        tr_prices, index=SCENARIOS, columns=hour_columns
    ),
    "NN: original + TR samples": pd.DataFrame(
        tr_nn_prices, index=SCENARIOS, columns=hour_columns
    ),
    "NN: original + equal global random samples": pd.DataFrame(
        new_nn_prices, index=SCENARIOS, columns=hour_columns
    ),
}
solutions = pd.concat(solution_frames, names=["method", "scenario"])
if solutions.shape != (4 * len(SCENARIOS), T):
    raise AssertionError(f"Unexpected solution table shape: {solutions.shape}.")

output_timestamp = time.strftime("%Y%m%d-%H%M%S")
summary_path = RESULTS_DIR / f"retraining_ablation_summary_{output_timestamp}.csv"
solutions_path = RESULTS_DIR / f"retraining_ablation_solutions_{output_timestamp}.csv"
comparison.reset_index().to_csv(summary_path, index=False)
solutions.to_csv(solutions_path)

display(comparison)
display(sample_count_table)
display(solutions)
print("Saved objective summary to:", summary_path)
print("Saved four solution matrices to:", solutions_path)

,true_objective,runtime_seconds,iterations,composite_queries,history_or_result_path
method,,,,,
Original NN,-98.480666,2223.791465,10000,NaN,/Users/liqi/Desktop/Codes/Surrogated-assisted ...
Trust region,-99.595549,11703.492167,24,1464.0,/Users/liqi/Desktop/Codes/Surrogated-assisted ...
NN: original + TR samples,-91.212067,2179.042543,10000,NaN,/Users/liqi/Desktop/Codes/Surrogated-assisted ...
NN: original + equal global random samples,-96.196163,2187.003335,10000,NaN,/Users/liqi/Desktop/Codes/Surrogated-assisted ...


,original_samples,trust_region_samples,new_random_samples,tr_combined_samples,new_combined_samples
scenario,,,,,
SB1,10000,1392,1392,11392,11392
SB2,10000,1342,1342,11342,11342
BG1,10000,1097,1097,11097,11097
BG2,10000,1064,1064,11064,11064
IP1,10000,1309,1309,11309,11309


hour_00   hour_01  \
method                                     scenario                       
Original NN                                SB1       0.970181  0.972706   
                                           SB2       0.984327  1.489236   
                                           BG1       1.499816  1.499887   
                                           BG2       1.499647  1.499705   
                                           IP1       1.496455  1.492518   
Trust region                               SB1       1.000002  0.991217   
                                           SB2       0.998743  1.457483   
                                           BG1       1.499985  1.499985   
                                           BG2       1.499987  1.499985   
                                           IP1       1.488347  1.478651   
NN: original + TR samples                  SB1       1.499712  1.215124   
                                           SB2       1.087324  1.011228   
                                           BG1       1.499889  1.499525   
                                           BG2       1.499655  1.499598   
                                           IP1       1.477153  1.348952   
NN: original + equal global random samples SB1       1.499040  0.974682   
                                           SB2       0.978410  1.168926   
                                           BG1       1.499913  1.499880   
                                           BG2       1.499779  1.499739   
                                           IP1       1.492821  1.495248   

                                                      hour_02   hour_03  \
method                                     scenario                       
Original NN                                SB1       0.975164  1.000694   
                                           SB2       0.971531  0.980285   
                                           BG1       1.499895  1.499872   
                                           BG2       1.499627  1.499876   
                                           IP1       1.494619  1.495886   
Trust region                               SB1       1.000069  0.999314   
                                           SB2       0.999609  1.000127   
                                           BG1       1.499986  1.499985   
                                           BG2       1.499987  1.499985   
                                           IP1       1.495236  1.488084   
NN: original + TR samples                  SB1       1.316121  1.388250   
                                           SB2       0.993166  0.986937   
                                           BG1       1.499817  1.499912   
                                           BG2       1.499785  1.499897   
                                           IP1       1.494497  1.492429   
NN: original + equal global random samples SB1       1.420878  1.024053   
                                           SB2       1.184377  0.975395   
                                           BG1       1.499895  1.499868   
                                           BG2       1.499837  1.499943   
                                           IP1       1.493362  1.493652   

                                                      hour_04   hour_05  \
method                                     scenario                       
Original NN                                SB1       0.959616  0.969702   
                                           SB2       0.946286  0.950465   
                                           BG1       1.499943  1.499931   
                                           BG2       1.499915  1.499792   
                                           IP1       1.491641  1.457786   
Trust region                               SB1       0.998336  0.997233   
                                           SB2       0.999625  0.999985   
                                           BG1       1.499984  1.499985   
  

Saved objective summary to: /Users/liqi/Desktop/Codes/Surrogated-assisted Stackelnerg Game/results/retraining_ablation_summary_20260919-183137.csv
Saved four solution matrices to: /Users/liqi/Desktop/Codes/Surrogated-assisted Stackelnerg Game/results/retraining_ablation_solutions_20260919-183137.csv
